# Aula 3: Criação de Features

## Análise RFV (Recência, Frequência, Valor)

A base das nossas features vem do framework RFV (ou RFM em inglês):
- **Recência**: Dias desde a última transação
    - "Quando foi a última vez que esse cliente apareceu?"
- **Frequência**: Nº de dias/transações distintos
    - "Com que frequência ele aparece?"
- **Valor**: Soma de pontos positivos
    - "Quanto ele engaja quando aparece?"

> Por que RFV é poderoso? \
> Porque captura três dimensões independentes do comportamento. Dois usuários podem ter a mesma frequência, mas um tem recência alta (ativo ontem) e outro baixa (sumiu há 60 dias). O RFV separa esses casos.


## A "foto" vs o "filme"
Um ponto crucial: essas features descrevem o usuário numa janela específica (uma "foto"). O alvo (churn) olha para frente (um "filme"). É essencial separar bem os dois:

```text
   ┌─────────────────┐  ┌────────────────┐
   │  JANELA (28d)   │  │  ALVO (churn)  │
   │  Features RFV   │→ │  Ativo ou não? │
   │  01/mai → 28/mai│  │  29/mai → ...  │
   └─────────────────┘  └────────────────┘
              ↑                      ↑
          "foto"                 "filme"
       (o que sei hoje)    (o que quero prever)
```




## 1. Definindo a janela de observação
Antes de escrever qualquer query, precisamos fixar as datas:
- Data de referência (`DtRef`) = '2026-06-01' → o "hoje" a partir do qual olhamos para trás.
- Janela de features = `DtRef` - 28 dias até `DtRef` → 28 dias de histórico.


### Pontos importantes
- `COUNT(DISTINCT DATE(DtCriacao))` → frequência em dias, não em transações. O usuário que fez 100 transações em 1 dia tem frequência = 1.
- `COUNT(*)` → frequência em transações. É o "F bruto".
- `SUM(CASE WHEN ...)` → separa pontos positivos (ganhos) de negativos (resgates). Isso é diferente de só olhar o saldo!
- `MIN(DATEDIFF(...))` → a recência é o menor intervalo, não o maior. Cuidado com a inversão lógica: quanto menor a recência, mais recente é o usuário (mais engajado).

In [0]:
%sql
-- =====================================================================
-- Bloco 1: Features RFV na janela de 28 dias
-- =====================================================================
-- Definição da janela:
--   • DtRef = '2026-06-01' (data de referência)
--   • Janela = DtRef - 28 dias até DtRef
--
-- Só consideramos transações DENTRO dessa janela.
-- O alvo (churn) será calculado FORA dessa janela (próxima aula).
--
-- Métricas calculadas (RFV + extensões):
--   • frequencia         → dias distintos com atividade (F)
--   • frequenciaTransacoes → nº total de transações (F bruto)
--   • valor / qntPontos_pos → pontos ganhos (V)
--   • qntPontos_neg      → pontos gastos (V negativo)
--   • qntPontos          → saldo líquido
--   • recencia           → dias desde a última transação até DtRef (R)
-- =====================================================================

SELECT 
    IdCliente,
    COUNT(DISTINCT DATE(DtCriacao)) AS frequencia,
    COUNT(*) AS frequenciaTransacoes,
    SUM(CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS valor,
    SUM(CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS qntPontos_pos,
    SUM(CASE WHEN QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qntPontos_neg,
    SUM(QtdePontos) AS qntPontos,
    -- Recência: dias desde a última transação até a data de referência
    -- Usamos MIN pois queremos o MENOR intervalo (transação mais recente)
    MIN(DATEDIFF('2026-06-01', DtCriacao)) AS recencia

FROM workspace.tmw_loyalty.transacoes
WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS
GROUP BY ALL;

## 2. Features temporais (share de dia/mês/semana)
Agora queremos capturar quando o usuário aparece. Um usuário que só aparece no fim de semana tem comportamento diferente de um que aparece todo dia.

> Por que isso importa para churn? \
> Se o padrão típico é "aparecer no fim de semana" e o usuário para de aparecer em fins de semana, isso é sinal precoce de churn, mesmo que a recência ainda esteja baixa.

In [0]:
%sql
-- =====================================================================
-- Bloco 2: Features temporais (dia da semana, mês, semana do ano)
-- =====================================================================
-- Objetivo: descrever QUANDO o usuário aparece.
-- Estas features ficam no nível (cliente, data) e depois agregamos.
--
-- Para construir os "shares" precisamos:
--   1. Extrair as partes de data (este bloco)
--   2. Agregar por cliente (próximo bloco)
-- =====================================================================

SELECT 
    IdCliente,
    DATE(DtCriacao) AS DtCriacao,
    DAYOFWEEK(DtCriacao) AS DiaDaSemana,  -- 1=Domingo, 7=Sábado
    MONTH(DtCriacao) AS Mes,               -- 1 a 12
    WEEKOFYEAR(DtCriacao) AS SemanaDoAno   -- 1 a 52

FROM workspace.tmw_loyalty.transacoes
WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS;


##  3. Share de transações por produto
Vimos na Aula 1 que temos transacao_produto e produtos. Um usuário que resgata muitos produtos diferentes é mais engajado do que um que só acumula.

In [0]:
%sql
-- Descobrindo os Top Produtos
SELECT 
    p.DescNomeProduto,
    COUNT(1) AS QtdeTransacoes,
    SUM(t.QtdePontos) AS QtdePontos 
FROM workspace.tmw_loyalty.transacoes AS t
LEFT JOIN workspace.tmw_loyalty.transacao_produto AS tp
    ON t.IdTransacao = tp.IdTransacao
LEFT JOIN workspace.tmw_loyalty.produtos AS p 
    ON p.IdProduto = tp.IdProduto
WHERE DATE(t.DtCriacao) < '2026-06-01'
GROUP BY ALL 
ORDER BY 2 DESC 
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- =====================================================================
-- Bloco 3: Share de transações e pontos por produto
-- =====================================================================
-- Objetivo: entender o "mix" de produtos de cada cliente.
--
-- Para cada cliente, calculamos:
--   • Quantos produtos distintos ele transacionou
--   • O share de transações de cada produto (top N produtos)
--   • O share de pontos gastos em cada produto
--
-- Estratégia:
--   1. Join transacoes ↔ transacao_produto ↔ produtos
--   2. Agregar por (cliente, produto)
--   3. Calcular shares por cliente
-- =====================================================================

WITH 
base AS (
    SELECT 
        t1.IdCliente,
        t2.IdProduto,
        t3.DescNomeProduto,
        t2.QtdeProduto,
        t2.vlProduto
    FROM workspace.tmw_loyalty.transacoes AS t1
    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
    WHERE t1.DtCriacao < '2026-06-01' AND t1.DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS
),

por_cliente_produto AS (
    SELECT
        IdCliente,
        DescNomeProduto,
        COUNT(*) AS qtdeTransacoes,
        SUM(QtdeProduto) AS qtdeProdutoTotal,
        SUM(vlProduto) AS vlProdutoTotal
    FROM base
    WHERE IdProduto IS NOT NULL
    GROUP BY ALL
),

por_cliente AS (
    SELECT
        IdCliente,
        COUNT(DISTINCT DescNomeProduto) AS qtdeProdutosDistintos,
        SUM(qtdeTransacoes) AS totalTransacoesComProduto,
        SUM(vlProdutoTotal) AS totalVlProduto
    FROM por_cliente_produto
    GROUP BY ALL
)

SELECT
    p.IdCliente,
    p.qtdeProdutosDistintos,
    p.totalTransacoesComProduto,
    p.totalVlProduto,
    -- Share por produto (top 3, por exemplo)
    ROUND(100.0 * SUM(CASE WHEN cp.DescNomeProduto = 'ChatMessage' THEN cp.qtdeTransacoes ELSE 0 END) / p.totalTransacoesComProduto, 2) AS share_ChatMessage_pct,
    ROUND(100.0 * SUM(CASE WHEN cp.DescNomeProduto = 'Lista de presença' THEN cp.qtdeTransacoes ELSE 0 END) / p.totalTransacoesComProduto, 2) AS share_Presenca_pct,
    ROUND(100.0 * SUM(CASE WHEN cp.DescNomeProduto = 'Presença Streak' THEN cp.qtdeTransacoes ELSE 0 END) / p.totalTransacoesComProduto, 2) AS share_Streak_pct,
    ROUND(100.0 * SUM(CASE WHEN cp.DescNomeProduto = 'Resgatar Ponei' THEN cp.qtdeTransacoes ELSE 0 END) / p.totalTransacoesComProduto, 2) AS share_Ponei_pct,
    ROUND(100.0 * SUM(CASE WHEN cp.DescNomeProduto = 'Troca de Pontos StreamElements' THEN cp.qtdeTransacoes ELSE 0 END) / p.totalTransacoesComProduto, 2) AS share_StreamElements_pct

FROM por_cliente_produto AS cp
JOIN por_cliente AS p
    ON p.IdCliente = cp.IdCliente
GROUP BY ALL;


## 4.Streaks e efeito rede
Essas são as features mais "comportamentais", e talvez as mais poderosas para churn.



In [0]:
%sql
-- =====================================================================
-- Bloco 4: Streaks (sequências) e efeito rede
-- =====================================================================
-- Streak = dias consecutivos de atividade.
--
-- Features:
--   • maiorStreak       → qual o maior streak na janela?
--   • streakAtual       → o streak terminou antes ou ainda está ativo?
--   • diasDesdeUltimoStreak → dias desde o último streak
--   • jaFezStreak (0/1) → flag binária
-- =====================================================================



Streaks em SQL puro são complexas (usam window functions). Vale discutir duas abordagens:

### A: Simples (contagem de dias consecutivos no fim da janela):

In [0]:
%sql
-- Streak "atual": quantos dias consecutivos de atividade terminando em DtRef
WITH 
dias_ativos AS (
    SELECT DISTINCT
        IdCliente,
        DATE(DtCriacao) AS DtDia
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS
),

-- Numera os dias de cada cliente para detectar quebras
numerados AS (
    SELECT
        IdCliente,
        DtDia,
        DATEDIFF(DtDia, LAG(DtDia) OVER (PARTITION BY IdCliente ORDER BY DtDia)) AS gap
    FROM dias_ativos
)

SELECT
    IdCliente,
    COUNT(*) AS diasTotaisAtivos,
    MAX(gap) AS maiorGap,
    COUNT(CASE WHEN gap = 1 THEN 1 END) AS diasConsecutivos,
    MAX(CASE WHEN gap = 1 THEN 1 ELSE 0 END) AS jaFezStreak
FROM numerados
GROUP BY ALL;

### B: Efeito rede (mais avançada):

O "efeito rede" mede como o comportamento do usuário se relaciona com o comportamento agregado dos outros. Exemplo: "no dia em que o usuário esteve ativo, quantos outros usuários estavam ativos também?"


> Um usuário com `mediaPopulacaoQuandoAtivo` alta é um usuário "social", aparece em dias movimentados. Isso pode indicar que ele é sensível a eventos coletivos (drops, lives especiais). Esse tipo de usuário pode ser mais fácil de reter com eventos.

In [0]:
%sql
-- Efeito rede: correlação entre atividade individual e coletiva
WITH 
atividade_diaria AS (
    SELECT
        DATE(DtCriacao) AS DtDia,
        IdCliente
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS
),

populacao_por_dia AS (
    SELECT
        DtDia,
        COUNT(DISTINCT IdCliente) AS qtdeUsuariosAtivos
    FROM atividade_diaria
    GROUP BY ALL
),

por_cliente AS (
    SELECT
        a.IdCliente,
        AVG(p.qtdeUsuariosAtivos) AS mediaPopulacaoQuandoAtivo
    FROM atividade_diaria AS a
    JOIN populacao_por_dia AS p
        ON p.DtDia = a.DtDia
    GROUP BY ALL
)

SELECT * 
FROM por_cliente
ORDER BY mediaPopulacaoQuandoAtivo DESC;

Databricks visualization. Run in Databricks to view.

## 5. Ranking padronizado (z-score)
Vimos no resumo da aula 4 que usamos a fórmula de padronização: X = (X - X_barra) / s

Isso transforma variáveis em z-scores, o que ajuda o modelo a comparar "importância relativa" independente da escala.

> **Pegadinha de MLOps**: na vida real, você não pode calcular a média/desvio com a população inteira, só com o treino. Senão há _data leakage_. Em produção, você salva a média e o desvio do treino e aplica nos novos dados. Esse é um tema da Aula 5 (SEMMA / Modify).

In [0]:
%sql
-- =====================================================================
-- Bloco 5: Ranking padronizado (z-score)
-- =====================================================================
-- Ideia: comparar o cliente com a média da população.
-- Um z-score de +2 significa "2 desvios-padrão acima da média".
--
-- Vamos padronizar a feature "valor" (pontos positivos).
-- =====================================================================

WITH 
rfv AS (
    SELECT 
        IdCliente,
        SUM(CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS valor
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - INTERVAL 28 DAYS
    GROUP BY ALL
),

estatisticas AS (
    SELECT
        AVG(valor) AS media,
        STDDEV(valor) AS desvio
    FROM rfv
)

SELECT
    r.IdCliente,
    r.valor,
    e.media,
    e.desvio,
    ROUND((r.valor - e.media) / e.desvio, 4) AS valor_zscore
FROM rfv AS r
CROSS JOIN estatisticas AS e;

       ┌──────────────────────────────────────────────┐
       │             FEATURES (RFV+)                  │
       │                                              │
       │  R  Recência       → dias sem aparecer       │
       │  F  Frequência     → dias/transações         │
       │  V  Valor          → pontos ganhos/gastos    │
       │                                              │
       │  + Temporal        → share dia/mês/semana    │
       │  + Produto         → mix de resgates         │
       │  + Streak          → sequências              │
       │  + Rede            → efeito coletivo         │
       │  + Ranking         → posição relativa        │
       └──────────────────────────────────────────────┘
                          ↓
                    [  MODELO  ]
                          ↓
       ┌──────────────────────────────────────────────┐
       │         ALVO: churn                          │
       │                                              │
       │  churn = 1 se não teve transação             │
       │           na janela seguinte (ex.: 28d)      │
       └──────────────────────────────────────────────┘

## Resumo do que discutimos

| Bloco | Feature | Pergunta que responde |
| --- | --- | --- |
| 1 | RFV | Quem é o cliente? |
| 2 | Share temporal | Quando ele aparece? |
| 3 | Share de produto | O que ele consome? |
| 4 | Streaks | Ele é consistente? |
| 5 | Efeito rede | Ele é social? |
| 6 | Z-score | Ele é "acima da média"? |

Todas essas features juntas formam a "foto" do cliente, e o modelo vai aprender a mapear essa foto para a probabilidade de churn.